In [1]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from pathlib import Path

In [2]:
project_dir = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)

file_path = project_dir / "data" / "processed" / "ted_talks_cleaned.csv"

df = pd.read_csv(file_path)

print("Cleaned dataset loaded successfully!")
print("Dataset shape:", df.shape)

display(df.head())

Cleaned dataset loaded successfully!
Dataset shape: (4005, 24)


,talk_id,title,speaker_1,all_speakers,occupations,about_speakers,views,recorded_date,published_date,event,...,topics,related_talks,url,description,transcript,title_clean,topics_clean,description_clean,transcript_clean,combined_text
0,1,Averting the climate crisis,Al Gore,{0: 'Al Gore'},{0: ['climate advocate']},{0: 'Nobel Laureate Al Gore focused the world’...,3523392,2006-02-25,2006-06-27,TED2006,...,"['alternative energy', 'cars', 'climate change...","{243: 'New thinking on the climate crisis', 54...",https://www.ted.com/talks/al_gore_averting_the...,With the same humor and humanity he exuded in ...,"Thank you so much, Chris. And it's truly a gre...",averting the climate crisis,"['alternative energy', 'cars', 'climate change...",with the same humor and humanity he exuded in ...,"thank you so much, chris. and it's truly a gre...",averting the climate crisis ['alternative ener...
1,92,The best stats you've ever seen,Hans Rosling,{0: 'Hans Rosling'},{0: ['global health expert; data visionary']},"{0: 'In Hans Rosling’s hands, data sings. Glob...",14501685,2006-02-22,2006-06-27,TED2006,...,"['Africa', 'Asia', 'Google', 'demo', 'economic...","{2056: ""Own your body's data"", 2296: 'A visual...",https://www.ted.com/talks/hans_rosling_the_bes...,You've never seen data presented like this. Wi...,"About 10 years ago, I took on the task to teac...",the best stats you've ever seen,"['africa', 'asia', 'google', 'demo', 'economic...",you've never seen data presented like this. wi...,"about 10 years ago, i took on the task to teac...","the best stats you've ever seen ['africa', 'as..."
2,7,Simplicity sells,David Pogue,{0: 'David Pogue'},{0: ['technology columnist']},{0: 'David Pogue is the personal technology co...,1920832,2006-02-24,2006-06-27,TED2006,...,"['computers', 'entertainment', 'interface desi...","{1725: '10 top time-saving tech tips', 2274: '...",https://www.ted.com/talks/david_pogue_simplici...,New York Times columnist David Pogue takes aim...,"(Music: ""The Sound of Silence,"" Simon & Garfun...",simplicity sells,"['computers', 'entertainment', 'interface desi...",new york times columnist david pogue takes aim...,"(music: ""the sound of silence,"" simon & garfun...","simplicity sells ['computers', 'entertainment'..."
3,53,Greening the ghetto,Majora Carter,{0: 'Majora Carter'},{0: ['activist for environmental justice']},{0: 'Majora Carter redefined the field of envi...,2664069,2006-02-26,2006-06-27,TED2006,...,"['MacArthur grant', 'activism', 'business', 'c...",{1041: '3 stories of local eco-entrepreneurshi...,https://www.ted.com/talks/majora_carter_greeni...,"In an emotionally charged talk, MacArthur-winn...",If you're here today — and I'm very happy that...,greening the ghetto,"['macarthur grant', 'activism', 'business', 'c...","in an emotionally charged talk, macarthur-winn...",if you're here today — and i'm very happy that...,"greening the ghetto ['macarthur grant', 'activ..."
4,66,Do schools kill creativity?,Sir Ken Robinson,{0: 'Sir Ken Robinson'},"{0: ['author', 'educator']}","{0: ""Creativity expert Sir Ken Robinson challe...",65051954,2006-02-25,2006-06-27,TED2006,...,"['children', 'creativity', 'culture', 'dance',...","{865: 'Bring on the learning revolution!', 173...",https://www.ted.com/talks/sir_ken_robinson_do_...,Sir Ken Robinson makes an entertaining and pro...,Good morning. How are you? (Audience) Good. It...,do schools kill creativity?,"['children', 'creativity', 'culture', 'dance',...",sir ken robinson makes an entertaining and pro...,good morning. how are you? (audience) good. it...,"do schools kill creativity? ['children', 'crea..."


In [3]:
print("Available columns:")
print(df.columns.tolist())

Available columns:
['talk_id', 'title', 'speaker_1', 'all_speakers', 'occupations', 'about_speakers', 'views', 'recorded_date', 'published_date', 'event', 'native_lang', 'available_lang', 'comments', 'duration', 'topics', 'related_talks', 'url', 'description', 'transcript', 'title_clean', 'topics_clean', 'description_clean', 'transcript_clean', 'combined_text']


In [4]:
required_columns = ["title", "combined_text"]

for column in required_columns:
    if column in df.columns:
        print(f"{column}: Available")
    else:
        print(f"{column}: Not available")

title: Available
combined_text: Available


In [5]:
df["combined_text"] = df["combined_text"].fillna("").astype(str)

usable_mask = df["combined_text"].str.strip().ne("")

usable_df = df.loc[usable_mask].copy()

print("Total records:", len(df))
print("Usable records:", len(usable_df))
print("Records with empty combined text:", len(df) - len(usable_df))

Total records: 4005
Usable records: 4005
Records with empty combined text: 0


In [6]:
tfidf_vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=10000
)

tfidf_matrix = tfidf_vectorizer.fit_transform(
    usable_df["combined_text"]
)

print("TF-IDF vectorization completed!")
print("TF-IDF matrix shape:", tfidf_matrix.shape)

TF-IDF vectorization completed!
TF-IDF matrix shape: (4005, 10000)


In [7]:
feature_names = tfidf_vectorizer.get_feature_names_out()

print("Total vocabulary terms:", len(feature_names))
print("\nFirst 30 vocabulary terms:")
print(feature_names[:30])

Total vocabulary terms: 10000

First 30 vocabulary terms:
['00' '000' '10' '100' '101' '10th' '11' '110' '11th' '12' '120' '125'
 '12th' '13' '130' '13th' '14' '140' '14th' '15' '150' '15th' '16' '160'
 '16th' '17' '170' '17th' '18' '180']


In [8]:
first_talk_vector = tfidf_matrix[0]

print("First talk vector shape:", first_talk_vector.shape)
print("Number of non-zero features:", first_talk_vector.nnz)

First talk vector shape: (1, 10000)
Number of non-zero features: 527


In [9]:
first_talk_dense = first_talk_vector.toarray().flatten()

top_word_indices = np.argsort(first_talk_dense)[::-1][:10]

top_words = [
    (feature_names[index], first_talk_dense[index])
    for index in top_word_indices
    if first_talk_dense[index] > 0
]

print("Top TF-IDF words in the first talk:")

for word, score in top_words:
    print(f"{word}: {score:.4f}")

Top TF-IDF words in the first talk:
carbon: 0.2314
laughter: 0.2263
climate: 0.1774
emissions: 0.1554
al: 0.1515
warming: 0.1504
restaurant: 0.1501
gore: 0.1474
trucks: 0.1356
efficiency: 0.1254


In [10]:
similarity_matrix = cosine_similarity(tfidf_matrix)

print("Similarity matrix shape:", similarity_matrix.shape)

Similarity matrix shape: (4005, 4005)


In [11]:
def recommend_talks(talk_index, top_n=5):
    similarity_scores = similarity_matrix[talk_index]

    similar_indices = np.argsort(similarity_scores)[::-1]

    similar_indices = [
        index for index in similar_indices
        if index != talk_index
    ]

    top_indices = similar_indices[:top_n]

    recommendations = usable_df.iloc[top_indices][
        ["talk_id", "title", "url"]
    ].copy()

    recommendations["similarity_score"] = [
        similarity_scores[index]
        for index in top_indices
    ]

    return recommendations

In [12]:
display(
    usable_df[["title", "speaker_1"]]
    .head(20)
)

,title,speaker_1
0,Averting the climate crisis,Al Gore
1,The best stats you've ever seen,Hans Rosling
2,Simplicity sells,David Pogue
3,Greening the ghetto,Majora Carter
4,Do schools kill creativity?,Sir Ken Robinson
5,Behind the design of Seattle's library,Joshua Prince-Ramus
6,Letting go of God,Julia Sweeney
7,Let's teach religion -- all religion -- in sch...,Dan Dennett
8,A life of purpose,Rick Warren
9,My wish: A global day of film,Jehane Noujaim


In [13]:
selected_indices = [0, 1, 2]

for index in selected_indices:
    print(f"\nSelected Talk {index + 1}:")
    print(usable_df.iloc[index]["title"])


Selected Talk 1:
Averting the climate crisis

Selected Talk 2:
The best stats you've ever seen

Selected Talk 3:
Simplicity sells


In [14]:
for index in selected_indices:
    selected_title = usable_df.iloc[index]["title"]

    print("\n" + "=" * 80)
    print("Selected TED Talk:", selected_title)
    print("=" * 80)

    recommendations = recommend_talks(index, top_n=5)

    display(recommendations)


Selected TED Talk: Averting the climate crisis


,talk_id,title,url,similarity_score
209,243,New thinking on the climate crisis,https://www.ted.com/talks/al_gore_new_thinking...,0.310985
2398,2784,A climate solution where all sides can win,https://www.ted.com/talks/ted_halstead_a_clima...,0.310260
3944,60080,How we could change the planet's climate future,https://www.ted.com/talks/david_wallace_wells_...,0.297766
2554,3579,Can we stop climate change by removing CO2 fro...,https://www.ted.com/talks/tim_kruger_can_we_st...,0.288179
3134,32560,The disarming case to act right now on climate...,https://www.ted.com/talks/greta_thunberg_the_d...,0.282813



Selected TED Talk: The best stats you've ever seen


,talk_id,title,url,similarity_score
492,620,Let my dataset change your mindset,https://www.ted.com/talks/hans_rosling_let_my_...,0.610112
114,140,New insights on poverty,https://www.ted.com/talks/hans_rosling_new_ins...,0.439384
745,974,The good news of the decade? We're winning the...,https://www.ted.com/talks/hans_rosling_the_goo...,0.381736
693,912,"Global population growth, box by box",https://www.ted.com/talks/hans_rosling_global_...,0.363452
438,540,"Insights on HIV, in stunning data visuals",https://www.ted.com/talks/hans_rosling_insight...,0.349857



Selected TED Talk: Simplicity sells


,talk_id,title,url,similarity_score
20,87,Nerdcore comedy,https://www.ted.com/talks/ze_frank_nerdcore_co...,0.336098
363,436,Design and discovery,https://www.ted.com/talks/david_carson_design_...,0.309859
404,495,Cool tricks your phone can do,https://www.ted.com/talks/david_pogue_cool_tri...,0.308818
1013,1268,6 ways to save the internet,https://www.ted.com/talks/roger_mcnamee_6_ways...,0.288601
93,114,A comic sendup of TED2006,https://www.ted.com/talks/tom_rielly_a_comic_s...,0.288512


## Manual Review of Sample Recommendations

I selected three TED Talks and generated the top five similar talks for each using TF-IDF and cosine similarity.

I reviewed the recommended titles and similarity scores to assess whether the recommendations appeared relevant to the selected talks.

The review focused on topic overlap, text similarity, and whether the recommended talks provided useful related content.

The observations below are based on the actual recommendation results generated by the notebook.


## Limitations of TF-IDF and Cosine Similarity

1. **Limited understanding of meaning:** TF-IDF mainly compares word usage. It may not fully understand the semantic meaning of a TED Talk.

2. **Different vocabulary:** Two talks about the same subject may receive a low similarity score if they use different words.

3. **Similar words but different meanings:** Talks that use similar words in different contexts may receive high similarity scores even when their meanings are not closely related.

4. **No guarantee of user preference:** A high cosine similarity score indicates textual similarity, but it does not guarantee that a user will enjoy or prefer the recommended talk.

5. **Vocabulary limitations:** TF-IDF does not naturally understand synonyms, word relationships, or contextual meaning.

6. **Computational cost:** Calculating a complete pairwise similarity matrix can require considerable memory and computation when the dataset becomes large.

## Observations

The observations from the actual recommendation results will be recorded after manually reviewing the recommendations for the three selected talks.